# Train and evaluate the project DQN on Google Colab

This notebook runs the upstream-compatible PyTorch profile: linked Atari preprocessing, Adam/Huber DQN, upstream replay sampling, epsilon and target schedules, periodic best-of-five evaluation, GIFs, and TensorBoard summaries. The network is ported to PyTorch and Gymnasium/ALE, so random-number sequences and runtime-level scores will not be byte-for-byte identical to the original TensorFlow/Gym code. Training stops at the first of the selected-action budget or the configured episode cap.

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU). Run cells from top to bottom. The Atari ROM distributed with ALE is subject to its own license; use it only where permitted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

GAME_NAME = 'Breakout'  # Choose Atlantis, Boxing, Breakout, or Pong
ENV_IDS = {name: f'ALE/{name}-v5' for name in ('Atlantis', 'Boxing', 'Breakout', 'Pong')}
ENV_ID = ENV_IDS[GAME_NAME]

PROJECT_DIR = Path('/content/DQN')
if not (PROJECT_DIR / 'pyproject.toml').exists():
    !git clone https://github.com/bhagathchan1/DQN.git {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull origin main
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

In [ ]:
import os, sys
from pathlib import Path
PROJECT_DIR = Path('/content/DQN')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
%pip install -e '.[env]'
import torch
from dqn_project.environment import make_env
print('Project package import: OK')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Choose Runtime → Change runtime type → T4 GPU, then rerun.'


In [ ]:
# Check Gymnasium + ALE + the upstream-style Atari pipeline before a long run.
import sys
sys.path.insert(0, str(PROJECT_DIR / 'src'))
%run scripts/environment_smoke_test.py --pipeline upstream --env $ENV_ID


## Prepare a reviewable transition sample

The linked DQN project generates its replay data online. This optional cell collects a small random-policy sample with the upstream-style Atari preprocessing pipeline and saves the transition arrays plus a four-frame visualization to Drive. It is a pipeline inspection artifact, not the agent's learned replay buffer.

In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/DQN')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
sample_path = DRIVE_DIR / 'data' / f'{GAME_NAME.lower()}_transition_sample.npz'
!python scripts/collect_transition_sample.py --env {ENV_ID} --steps 256 --seed 7 --output {sample_path}
print('Sample archive:', sample_path)
print('Preview:', sample_path.with_name(sample_path.stem + '_first_state.png'))

In [ ]:
# Keep durable files on Drive. Update RUN_ID to create a separate experiment.
from pathlib import Path
import shutil, yaml

DRIVE_DIR = Path('/content/drive/MyDrive/DQN')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for name in ('checkpoints', 'results'):
    local = PROJECT_DIR / name
    durable = DRIVE_DIR / name
    durable.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        shutil.move(str(local), str(durable / 'initial_repository_artifacts'))
    local.symlink_to(durable, target_is_directory=True)

RUN_ID = f'{GAME_NAME.lower()}_upstream_port_colab_seed7_max10kepisodes'
config = yaml.safe_load((PROJECT_DIR / 'configs/upstream_breakout.yaml').read_text())
config['run']['id'] = RUN_ID
config['environment']['id'] = ENV_ID
config['training']['device'] = 'auto'
# Stop at the first of this decision budget or 10,000 completed episodes.
config['training']['total_environment_steps'] = 10_000_000
config['training']['max_episodes'] = 10_000
config_path = PROJECT_DIR / 'configs/colab_upstream.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print('Config:', config_path)
print('Checkpoints and metrics will persist in:', DRIVE_DIR)

In [ ]:
# Train. CUDA accelerates the network; the Atari environment and replay sampling also use CPU.
!dqn-train --config configs/colab_upstream.yaml

In [ ]:
# View the upstream-style episode summaries after training.
tensorboard_logdir = DRIVE_DIR / 'results/tensorboard'
%load_ext tensorboard
%tensorboard --logdir $tensorboard_logdir

In [ ]:
# Evaluate the final checkpoint, save a GIF, and display it in the notebook.
checkpoint = DRIVE_DIR / 'checkpoints' / f'{RUN_ID}_final.pt'
evaluation = DRIVE_DIR / 'results/metrics' / f'{RUN_ID}_evaluation.json'
video = DRIVE_DIR / 'results/evaluations' / f'{RUN_ID}_final_best.gif'
!dqn-evaluate --config configs/colab_upstream.yaml --checkpoint {checkpoint} --output {evaluation} --video {video}
from IPython.display import Image, display
display(Image(filename=str(video)))
print('Evaluation:', evaluation)

## Inspect the run

Training episode rewards and TensorBoard logs are in `MyDrive/DQN/results/`. The final weights are in `MyDrive/DQN/checkpoints/`; evaluation GIFs are under `results/evaluations/` and display above. This notebook uses the upstream-compatible PyTorch/Gymnasium profile and caps training at 10,000 episodes or 10 million selected actions, whichever comes first. Its scores can differ from the original TensorFlow/Gym code. The GIFs and figures under `assets/upstream_dqn/` remain upstream reference assets, separate from newly generated videos.